# Stopword Impact Study

**NLP Traditional Preprocessing Project — Stopword Removal Analysis**

## Project Overview

Stopword removal is one of the most common NLP preprocessing steps, yet its impact varies by task. We compare four stopword configurations (no removal, sklearn, NLTK, custom keeping negations) on binary sentiment classification using the IMDB dataset.

## Learning Objectives

1. Understand stopwords and why they are removed.
2. Compare stopword lists (sklearn, NLTK) and their coverage.
3. Quantify impact on accuracy and F1.
4. Learn when removal helps vs hurts.
5. Understand why negation words matter for sentiment.

## Problem Statement

**Does removing stopwords improve or hurt NLP model performance?** Controlled experiments on sentiment classification.

## Why This Project Matters

- Stopword removal is often applied blindly.
- Removing negation words flips meaning ("not good" -> "good").
- Vocabulary reduction saves memory.
- Understanding the trade-off prevents degrading quality.

## Dataset Overview

IMDB reviews, 5K subset, binary sentiment, ~50/50 balanced.

## Dataset Source and License Notes

Hugging Face `imdb`. Free for research. Auto-download.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas","numpy","matplotlib","seaborn","scikit-learn","nltk","datasets"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
import nltk; nltk.download("stopwords",quiet=True)
print("Ready.")

## Imports

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, re, warnings
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords as nltk_stopwords
from datasets import load_dataset
warnings.filterwarnings("ignore"); sns.set_style("whitegrid")
%matplotlib inline
print("Loaded.")

## Configuration / Constants

In [ ]:
SEED=42; N_SAMPLES=5000; TEST_SIZE=0.2; MAX_FEATURES=10000
CLASS_NAMES=["Negative","Positive"]; np.random.seed(SEED)

## Dataset Download or Loading

In [ ]:
ds=load_dataset("imdb",split="train")
df=ds.to_pandas().sample(n=N_SAMPLES,random_state=SEED).reset_index(drop=True)
print(f"Shape: {df.shape}"); df.head(3)

## Data Validation Checks

In [ ]:
print("Missing:",df.isnull().sum().to_dict())
n_dupes=df.duplicated(subset=["text"]).sum()
if n_dupes>0: df=df.drop_duplicates(subset=["text"]).reset_index(drop=True)
assert df["label"].isin([0,1]).all()
df["text_len"]=df["text"].str.len(); df["word_count"]=df["text"].str.split().apply(len)
print("Validation passed.")

## Exploratory Data Analysis

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(df["text_len"],bins=50,color="steelblue",edgecolor="white")
axes[0].set_title("Text Length")
for l,n in enumerate(CLASS_NAMES):
    axes[1].hist(df[df["label"]==l]["word_count"],bins=30,alpha=0.5,label=n)
axes[1].set_title("Words by Sentiment"); axes[1].legend()
plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
cc=df["label"].value_counts().sort_index()
fig,ax=plt.subplots(figsize=(6,4))
ax.bar(CLASS_NAMES,cc.values,color=["#C44E52","#55A868"])
ax.set_title("Sentiment Distribution")
plt.tight_layout(); plt.show()

## Train/Validation/Test Split Strategy

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df["text"],df["label"],test_size=TEST_SIZE,random_state=SEED,stratify=df["label"])
print(f"Train:{len(X_train)}|Test:{len(X_test)}")

## Preprocessing Strategy - Stopword Lists

We compare: No removal, sklearn stopwords, NLTK stopwords, Custom (keep negations).

In [ ]:
sklearn_stops=list(ENGLISH_STOP_WORDS); nltk_stops=nltk_stopwords.words("english")
NEGATION_WORDS={"not","no","never","nor","neither","nobody","nothing","nowhere","cannot","won","wouldn","shouldn","couldn","didn","doesn","hadn","hasn","haven","isn","aren","wasn","weren"}
custom_stops=[w for w in sklearn_stops if w not in NEGATION_WORDS]
STOPWORD_CONFIGS={"No removal":None,"sklearn stopwords":sklearn_stops,"NLTK stopwords":nltk_stops,"Custom (keep negations)":custom_stops}
print(f"sklearn:{len(sklearn_stops)}, NLTK:{len(nltk_stops)}, Custom:{len(custom_stops)}")

In [ ]:
s1,s2=set(sklearn_stops),set(nltk_stops)
print(f"Only sklearn:{len(s1-s2)}, Only NLTK:{len(s2-s1)}, Both:{len(s1&s2)}")

## Feature Engineering - Vocabulary Impact

In [ ]:
vs=[]
for cn,sw in STOPWORD_CONFIGS.items():
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,stop_words=sw,ngram_range=(1,2))
    X_tr=tfidf.fit_transform(X_train)
    vs.append({"Config":cn,"Features":X_tr.shape[1],"Nonzero/Doc":round(X_tr.nnz/X_tr.shape[0],1)})
print(pd.DataFrame(vs).to_string(index=False))

## Baseline Model

In [ ]:
tfidf_b=TfidfVectorizer(max_features=MAX_FEATURES,stop_words=None,ngram_range=(1,2))
X_tr_b=tfidf_b.fit_transform(X_train); X_te_b=tfidf_b.transform(X_test)
bl=MultinomialNB(); bl.fit(X_tr_b,y_train); y_pb=bl.predict(X_te_b)
bl_acc=accuracy_score(y_test,y_pb); bl_f1=f1_score(y_test,y_pb)
print(f"BASELINE: Acc={bl_acc:.4f}, F1={bl_f1:.4f}")
print(classification_report(y_test,y_pb,target_names=CLASS_NAMES))

## Full Stopword Config x Classifier Comparison

In [ ]:
classifiers={"MultinomialNB":MultinomialNB(),"LogisticRegression":LogisticRegression(max_iter=1000,random_state=SEED),"LinearSVC":LinearSVC(max_iter=2000,random_state=SEED)}
results=[]
for cn,sw in STOPWORD_CONFIGS.items():
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,stop_words=sw,ngram_range=(1,2))
    X_tr=tfidf.fit_transform(X_train); X_te=tfidf.transform(X_test)
    for clf_n,clf in classifiers.items():
        m=clone(clf); m.fit(X_tr,y_train); yp=m.predict(X_te)
        results.append({"Config":cn,"Classifier":clf_n,"Accuracy":accuracy_score(y_test,yp),"Precision":precision_score(y_test,yp),"Recall":recall_score(y_test,yp),"F1":f1_score(y_test,yp)})
results_df=pd.DataFrame(results)
print(results_df.sort_values("F1",ascending=False).to_string(index=False))

In [ ]:
pivot=results_df.pivot(index="Config",columns="Classifier",values="F1")
fig,ax=plt.subplots(figsize=(8,5))
sns.heatmap(pivot,annot=True,fmt=".4f",cmap="YlGnBu",ax=ax)
ax.set_title("F1: Stopword Config x Classifier")
plt.tight_layout(); plt.show()

## LazyPredict Benchmark

> **Note:** Not used for NLP tasks per best practices.

## FLAML AutoML

> **Note:** Not used for NLP tasks per best practices.

## Additional Best-Library Workflow - Stopword Importance Analysis

In [ ]:
tfidf_f=TfidfVectorizer(max_features=MAX_FEATURES,stop_words=None,ngram_range=(1,2))
X_tr_f=tfidf_f.fit_transform(X_train)
lr=LogisticRegression(max_iter=1000,random_state=SEED); lr.fit(X_tr_f,y_train)
fnames=tfidf_f.get_feature_names_out(); coefs=lr.coef_[0]
sw_feats=[]
for sw in sklearn_stops:
    if sw in fnames:
        idx=list(fnames).index(sw)
        sw_feats.append({"word":sw,"coef":coefs[idx],"abs":abs(coefs[idx])})
sw_df=pd.DataFrame(sw_feats).sort_values("abs",ascending=False)
print("Top 15 most important stopwords:")
print(sw_df.head(15).to_string(index=False))

## Top 3 Model Selection

In [ ]:
top3=results_df.sort_values("F1",ascending=False).head(3).reset_index(drop=True)
print(top3[["Config","Classifier","Accuracy","F1"]].to_string(index=False))
print(f"Baseline F1={bl_f1:.4f}, Best improvement: {((top3.iloc[0]['F1']-bl_f1)/bl_f1)*100:+.2f}%")

## Final Training and Evaluation of Top 3

In [ ]:
top3_preds={}
for i,row in top3.iterrows():
    sw=STOPWORD_CONFIGS[row["Config"]]
    tfidf=TfidfVectorizer(max_features=MAX_FEATURES,stop_words=sw,ngram_range=(1,2))
    X_tr=tfidf.fit_transform(X_train); X_te=tfidf.transform(X_test)
    if row["Classifier"]=="MultinomialNB": m=MultinomialNB()
    elif row["Classifier"]=="LogisticRegression": m=LogisticRegression(max_iter=1000,random_state=SEED)
    else: m=LinearSVC(max_iter=2000,random_state=SEED)
    m.fit(X_tr,y_train); yp=m.predict(X_te)
    top3_preds[f"{row['Config']}+{row['Classifier']}"]=yp
    print(f"\n{'='*60}\n#{i+1}\n{'='*60}")
    print(classification_report(y_test,yp,target_names=CLASS_NAMES))

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(18,5))
for idx,(n,yp) in enumerate(top3_preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test,yp),display_labels=CLASS_NAMES).plot(ax=axes[idx],cmap="Blues",values_format="d")
    axes[idx].set_title(f"#{idx+1}",fontsize=9)
plt.tight_layout(); plt.show()

## Error Analysis

In [ ]:
bp=list(top3_preds.values())[0]
em=bp!=y_test.values
edf=pd.DataFrame({"text":X_test.values[em],"true":y_test.values[em],"pred":bp[em]})
edf["true_class"]=edf["true"].map(dict(enumerate(CLASS_NAMES)))
edf["pred_class"]=edf["pred"].map(dict(enumerate(CLASS_NAMES)))
print(f"Errors: {len(edf)}/{len(y_test)} ({len(edf)/len(y_test)*100:.1f}%)")

## Interpretation and Business Insight

1. Stopword removal has modest impact on sentiment classification.
2. Keeping negation words helps sentiment analysis.
3. Vocabulary drops ~20-30% with stopword removal.
4. The effect is task-dependent.

## Limitations

Single dataset, English only, TF-IDF only, static lists, binary sentiment.

## How to Improve This Project

1. Multiple datasets. 2. Domain stopwords. 3. spaCy stopwords. 4. Topic classification test.

## Production Considerations

Task-specific lists. Domain stopwords. Consistency. Multilingual support.

## Common Mistakes

1. Removing negation words. 2. One list for all tasks. 3. Not benchmarking. 4. Over-removing.

## Mini Challenge / Exercises

1. Data-driven stopword list. 2. Test on 20 Newsgroups. 3. Negation-aware preprocessing.

## Final Summary / Key Takeaways

1. Don't blindly remove stopwords. 2. Keep negation for sentiment. 3. Impact is often small. 4. Build domain-specific lists.